# VITAL: Elder Safety HAR training (YOLO11, Colab GPU)
Classes: **0 Walking, 1 Standing, 2 Sitting, 3 Sleeping, 4 Falling**.
**FIXED VERSION:** reduces false 'Falling' (balanced Falling data, 'Sit Down' kept as Sitting, stronger classification loss, tuned Falling threshold, temporal filter for video).
Run cells **top to bottom**. After each cell read the *Expect / Stop if* note before continuing.
Runtime > Change runtime type > **T4 GPU**.

### Cell 1: install and import
**Does:** installs Ultralytics and checks the GPU.
**Expect:** an Ultralytics summary and `CUDA available: True`.
**Stop if:** `CUDA available: False` (switch the runtime to GPU).

In [ ]:
!pip install -q ultralytics
import torch, ultralytics
ultralytics.checks()
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Runtime > Change runtime type > T4 GPU, then re-run.")


### Cell 2: connect Google Drive (optional)
**Does:** mounts Drive so you can load the ZIP from it and save a backup of `best.pt`.
**Expect:** a permission popup, then `Mounted at /content/drive`. Skip if you upload the ZIP directly.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Cell 3: extract the dataset
**Does:** checks the ZIP is complete and extracts it to `/content/raw_ds` (the original, never modified later).
**Expect:** ZIP size, then a list of folders such as `train valid test data.yaml`.
**Stop if:** *not a valid ZIP* or *corrupted file* (upload again, Drive is more reliable).

In [ ]:
import zipfile, shutil
from pathlib import Path

# EDIT THIS: path of the dataset ZIP inside Colab.
#   Uploaded directly : /content/Human Action Recognition.v2i.yolov11.zip
#   From Google Drive : /content/drive/MyDrive/Human Action Recognition.v2i.yolov11.zip
ZIP_PATH = Path("/content/drive/MyDrive/Human Action Recognition.v2i.yolov11.zip")
RAW_DIR = Path("/content/raw_ds")          # original data is extracted here (never modified)
if not ZIP_PATH.exists():
    found = list(Path("/content").glob("*.zip"))
    drive_dir = Path("/content/drive/MyDrive")
    if drive_dir.exists():
        found += list(drive_dir.glob("*.zip"))
    raise FileNotFoundError(
        f"ZIP not found: {ZIP_PATH}\nZIP files I can see: {[str(f) for f in found] or 'none'}\n"
        "Upload the ZIP (folder icon) or mount Drive (Cell 2) and fix ZIP_PATH.")

print(f"ZIP size: {ZIP_PATH.stat().st_size / 1e6:.1f} MB (compare with the size on your PC)")
if not zipfile.is_zipfile(ZIP_PATH):
    raise RuntimeError("This is not a valid ZIP: the upload was probably cut off. "
                       "Delete it and upload again (Google Drive is more reliable).")

with zipfile.ZipFile(ZIP_PATH) as z:
    bad = z.testzip()
    if bad:
        raise RuntimeError(f"Corrupted file inside ZIP: {bad}. Re-download the dataset.")
    if RAW_DIR.exists():
        shutil.rmtree(RAW_DIR)
    z.extractall(RAW_DIR)

print("Extracted to", RAW_DIR)
for p in sorted(RAW_DIR.iterdir()):
    print("  ", p.name + ("/" if p.is_dir() else ""))


### Cell 4: dataset analysis (read-only)
**Does:** finds train/valid/test, reads real class names and IDs from `data.yaml`, counts images and annotations per class, checks corrupted images, missing/empty labels, invalid class IDs, bad or out-of-range boxes, image sizes, exact and near duplicates (including train/val leakage), tells you whether a waving class exists, and shows annotated samples.
**Expect:** a full printed report plus a grid of sample images.
**Stop if:** required classes are missing, or the samples show wrong labels. Copy the report here if unsure.

In [ ]:
# READ-ONLY analysis of the original dataset. Nothing is modified.
import json, hashlib, random
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np, cv2, yaml
from PIL import Image
import matplotlib.pyplot as plt

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# ---- 1. find structure automatically ----
ymls = sorted(RAW_DIR.rglob("data.yaml"))
if not ymls:
    raise FileNotFoundError(f"No data.yaml under {RAW_DIR}. Is this a YOLO-format export?")
YAML_PATH = ymls[0]
ROOT = YAML_PATH.parent
try:
    CFG = yaml.safe_load(YAML_PATH.read_text())
    names = CFG["names"]
except Exception as e:
    raise ValueError(f"Invalid data.yaml ({YAML_PATH}): {e}")
ORIG_NAMES = [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)
NC = len(ORIG_NAMES)

def find_splits(root):
    out = {}
    for key, cands in {"train": ["train"], "val": ["valid", "val", "validation"], "test": ["test"]}.items():
        out[key] = None
        for c in cands:
            if (root / c / "images").is_dir():
                out[key] = (root / c / "images", root / c / "labels"); break
            if (root / "images" / c).is_dir():
                out[key] = (root / "images" / c, root / "labels" / c); break

    return out
SPLITS = {k: v for k, v in find_splits(ROOT).items() if v}
print("Dataset root:", ROOT)
print("data.yaml   :", YAML_PATH)
for k in ("train", "val", "test"):
    print(f"  {k:5s}:", SPLITS.get(k) or "NOT FOUND")
if "train" not in SPLITS or "val" not in SPLITS:
    raise RuntimeError("train and valid/val folders are required but were not found.")

print("\n=== ORIGINAL CLASSES (read from data.yaml) ===")
for i, n in enumerate(ORIG_NAMES):
    print(f"  id {i:2d} = {n}")

# ---- helpers shared with the cleaning cell ----
def fix_box(xc, yc, w, h, tol=0.02, min_size=0.005):
    # returns a safe (xc,yc,w,h) or None if the box is unusable
    if not np.all(np.isfinite([xc, yc, w, h])):
        return None
    x1, y1, x2, y2 = xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2
    if min(x1, y1) < -tol or max(x2, y2) > 1 + tol:
        return None                      # far outside the image: suspicious, reject
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(1, x2), min(1, y2)   # tiny overflow: clip
    nw, nh = x2 - x1, y2 - y1
    if nw < min_size or nh < min_size:
        return None
    return ((x1 + x2) / 2, (y1 + y2) / 2, nw, nh)

def parse_label(path):
    boxes, issues = [], []
    try:
        lines = Path(path).read_text().splitlines()
    except Exception:
        return boxes, ["unreadable_label"]
    for line in lines:
        p = line.split()
        if not p:
            continue
        if len(p) != 5:
            issues.append("bad_format"); continue
        try:
            if not float(p[0]).is_integer():
                raise ValueError
            cid = int(float(p[0])); xc, yc, w, h = map(float, p[1:])
        except ValueError:
            issues.append("bad_format"); continue
        if cid < 0 or cid >= NC:
            issues.append("invalid_class_id"); continue
        if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1):
            issues.append("coords_outside_0_1")
        if fix_box(xc, yc, w, h) is None:
            issues.append("invalid_bbox"); continue
        boxes.append((cid, xc, yc, w, h))
    return boxes, issues

def dhash(im):
    g = np.asarray(im.convert("L").resize((9, 8)), dtype=np.int16)
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    return int.from_bytes(np.packbits(bits).tobytes(), "big")

_POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def nearest_hamming(q_hashes, ref_hashes):
    q = np.array(q_hashes, dtype=np.uint64); r = np.array(ref_hashes, dtype=np.uint64)
    d_out = np.full(len(q), 99, dtype=np.int32)
    if len(r) == 0 or len(q) == 0:
        return d_out
    for k in range(len(q)):
        x = np.bitwise_xor(r, q[k])
        d = _POP[x.view(np.uint8)].reshape(-1, 8).sum(1)
        d_out[k] = d.min()
    return d_out

# ---- 2. scan every image + label ----
RECORDS = []
orphan_labels = Counter()
for split, (imgd, lbld) in SPLITS.items():
    imgs = [p for p in sorted(imgd.iterdir()) if p.suffix.lower() in IMG_EXT]
    stems = {p.stem for p in imgs}
    if lbld.is_dir():
        orphan_labels[split] = sum(1 for p in lbld.glob("*.txt") if p.stem not in stems)
    for img in imgs:
        r = dict(split=split, img=img, lbl=lbld / (img.stem + ".txt"), w=0, h=0,
                 issues=[], boxes=[], dhash=None, md5=None)
        try:
            with Image.open(img) as im:
                im.verify()
            with Image.open(img) as im:
                r["w"], r["h"] = im.size
                r["dhash"] = dhash(im)
            r["md5"] = hashlib.md5(img.read_bytes()).hexdigest()
        except Exception:
            r["issues"].append("corrupted_image")
        if not r["lbl"].exists():
            r["issues"].append("missing_label")
        else:
            boxes, iss = parse_label(r["lbl"])
            r["boxes"], r["issues"] = boxes, r["issues"] + iss
            if not boxes and not iss:
                r["issues"].append("empty_label")
        RECORDS.append(r)

# ---- 3. report ----
print("\n=== IMAGES PER SPLIT ===")
for s in SPLITS:
    print(f"  {s:5s}: {sum(1 for r in RECORDS if r['split'] == s)} images | orphan label files: {orphan_labels[s]}")

print("\n=== ANNOTATIONS / IMAGES PER ORIGINAL CLASS ===")
ann = {s: Counter() for s in SPLITS}; imc = {s: defaultdict(set) for s in SPLITS}
for r in RECORDS:
    for (cid, *_rest) in r["boxes"]:
        ann[r["split"]][cid] += 1; imc[r["split"]][cid].add(str(r["img"]))
print(f"  {'class':22s}" + "".join(f"{s+' ann/img':>16s}" for s in SPLITS))
for i, n in enumerate(ORIG_NAMES):
    print(f"  {i:2d} {n:19s}" + "".join(f"{ann[s][i]:>8d}/{len(imc[s][i]):<7d}" for s in SPLITS))

print("\n=== ISSUES FOUND ===")
issue_count = Counter(i for r in RECORDS for i in set(r["issues"]))
if not issue_count:
    print("  none")
for k, v in issue_count.items():
    ex = [r["img"].name for r in RECORDS if k in r["issues"]][:3]
    print(f"  {k:20s}: {v} images   e.g. {ex}")

print("\n=== IMAGE SIZES ===")
sizes = Counter((r["w"], r["h"]) for r in RECORDS if r["w"])
print("  most common:", sizes.most_common(5))
odd = [r for r in RECORDS if r["w"] and (min(r["w"], r["h"]) < 100 or max(r["w"], r["h"]) / min(r["w"], r["h"]) > 3)]
print(f"  unusual (tiny or extreme aspect ratio): {len(odd)}")

print("\n=== DUPLICATES / LEAKAGE ===")
by_md5 = defaultdict(list)
for r in RECORDS:
    if r["md5"]: by_md5[r["md5"]].append(r)
exact = [g for g in by_md5.values() if len(g) > 1]
cross = [g for g in exact if len({x["split"] for x in g}) > 1]
print(f"  exact duplicate groups: {len(exact)} (across splits: {len(cross)})")
ok = {s: [r for r in RECORDS if r["split"] == s and r["dhash"] is not None] for s in SPLITS}
for later, earlier in (("val", ["train"]), ("test", ["train", "val"])):
    if later in SPLITS:
        ref = [r["dhash"] for s in earlier if s in ok for r in ok[s]]
        d = nearest_hamming([r["dhash"] for r in ok[later]], ref)
        print(f"  {later} images nearly identical to {'/'.join(earlier)} (hash distance<=3): {int((d <= 3).sum())} of {len(d)}")
print("  (Roboflow frames often come from videos -> neighbouring frames in train and val inflate scores."
      " The cleaning cell removes such val/test images.)")

print("\n=== IS THERE A WAVING CLASS? ===")
wavy = [n for n in ORIG_NAMES if any(k in n.lower() for k in ("wav", "hand", "gesture", "help"))]
print("  Waving-related classes:", wavy if wavy else "NONE -> waving cannot be learned from this dataset.")

# ---- 4. annotated samples ----
def draw_sample(r, names=ORIG_NAMES):
    im = cv2.cvtColor(cv2.imread(str(r["img"])), cv2.COLOR_BGR2RGB)
    H, W = im.shape[:2]
    for (cid, xc, yc, w, h) in r["boxes"]:
        x1, y1, x2, y2 = int((xc - w / 2) * W), int((yc - h / 2) * H), int((xc + w / 2) * W), int((yc + h / 2) * H)
        cv2.rectangle(im, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(im, f"{cid}:{names[cid]}", (x1, max(y1 - 5, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)
    return im
good = [r for r in RECORDS if r["boxes"] and "corrupted_image" not in r["issues"]]
pick = random.sample(good, min(12, len(good)))
fig, axs = plt.subplots(3, 4, figsize=(16, 10))
for ax, r in zip(axs.flat, pick):
    ax.imshow(draw_sample(r)); ax.set_title(f"{r['split']}/{r['img'].name[:22]}", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()


### Cell 5: clean and remap into a NEW dataset
**Does:** maps original classes (by NAME, never by assumed ID) into the 5 final classes, removes unwanted annotations, clips tiny box overflows and rejects bad boxes, drops corrupted or label-less images, removes val/test images that are near-copies of train (leakage), keeps the train/val/test separation, and writes `/content/VITAL_HAR_YOLO11_CLEAN`.
**Images with no required class left are dropped.** They contain people we would otherwise teach the model to treat as background.
**Expect:** a class mapping table, a cleaning summary and `cleaning_log.json`.
**Stop if:** it raises *required classes NOT found* (edit `ALIASES`).

In [ ]:
# Builds a NEW clean dataset. The original in RAW_DIR is never touched.
import shutil, json
from collections import Counter

FINAL_CLASSES = ["Walking", "Standing", "Sitting", "Sleeping", "Falling"]   # final IDs 0..4
DST = Path("/content/VITAL_HAR_YOLO11_CLEAN")

SIT_DOWN_AS_SITTING = True          # FIX: 'Sit Down' frames (bending/lowering into a chair) were deleted before, so the model
                                    # had no class for sit<->stand transitions and called them Falling. Now they are Sitting.
BALANCE_FALLING = True              # FIX: Falling was ~38% of the data (9x Sleeping) -> model over-predicted it
FALLING_CAP_FACTOR = 1.0            # Falling annotations are capped at this x (largest NON-falling class). Lower = fewer false falls
DEDUP_TRAIN_EXACT = True            # drop exact duplicate images inside train
INCLUDE_STAIRS_AS_WALKING = True    # 'Walking_on_Stairs' is still walking. Set False to drop it.
REMOVE_LEAKAGE = True               # drop val/test images that are near-copies of train (video frames)
LEAK_THRESHOLD = 3                  # dHash distance (0 = identical, 64 = different). 3 = near-identical.

def norm(n):
    return n.strip().lower().replace("-", "_").replace(" ", "_")

# original-name -> final-name. IDs are NOT assumed: they are looked up from the real data.yaml.
ALIASES = {
    "walking": "Walking", "walk": "Walking",
    "standing": "Standing", "stand": "Standing",
    "sitting": "Sitting", "sit": "Sitting",
    "lying_down": "Sleeping", "lying": "Sleeping", "laying": "Sleeping", "sleeping": "Sleeping",
    "fall_down": "Falling", "fall_detected": "Falling", "fallen": "Falling", "falling": "Falling", "fall": "Falling",
}
if INCLUDE_STAIRS_AS_WALKING:
    ALIASES["walking_on_stairs"] = "Walking"
if SIT_DOWN_AS_SITTING:
    ALIASES["sit_down"] = "Sitting"

ID_MAP, removed_classes = {}, []
print("=== CLASS MAPPING (original id -> final id) ===")
for oid, n in enumerate(ORIG_NAMES):
    t = ALIASES.get(norm(n))
    if t:
        ID_MAP[oid] = FINAL_CLASSES.index(t)
        print(f"  {oid:2d} {n:20s} -> {ID_MAP[oid]} {t}")
    else:
        removed_classes.append(n)
        print(f"  {oid:2d} {n:20s} -> REMOVED")
missing = [c for c in FINAL_CLASSES if c not in {FINAL_CLASSES[v] for v in ID_MAP.values()}]
if missing:
    raise ValueError(f"These required classes were NOT found in the dataset: {missing}\n"
                     f"Original classes: {ORIG_NAMES}\nEdit ALIASES above to map the right names.")

# ---- filter every record ----
stats, kept_by_split, rejected = Counter(), {s: [] for s in SPLITS}, []
for r in RECORDS:
    if "corrupted_image" in r["issues"]:
        rejected.append((r, "corrupted_image")); continue
    if "missing_label" in r["issues"]:
        rejected.append((r, "missing_label")); continue
    new = []
    for (cid, xc, yc, w, h) in r["boxes"]:
        if cid not in ID_MAP:
            stats["unwanted_annotations_removed"] += 1; continue
        fb = fix_box(xc, yc, w, h)
        if fb is None:
            stats["invalid_boxes_rejected"] += 1; continue
        new.append((ID_MAP[cid],) + fb)
    if not new:
        # Images with only unwanted classes contain people we would have to label as 'background'.
        # That teaches the model to ignore real people, so these images are dropped.
        rejected.append((r, "empty_label" if "empty_label" in r["issues"] else "no_required_class_left")); continue
    r["new"] = new
    kept_by_split[r["split"]].append(r)

# ---- remove cross-split leakage (train has priority, then val, then test) ----
if REMOVE_LEAKAGE:
    for later, earlier in (("val", ["train"]), ("test", ["train", "val"])):
        if later not in kept_by_split:
            continue
        ref = [r["dhash"] for s in earlier if s in kept_by_split for r in kept_by_split[s]]
        q = kept_by_split[later]
        d = nearest_hamming([r["dhash"] for r in q], ref)
        keep = []
        for r, dd in zip(q, d):
            (rejected.append((r, "near_duplicate_of_earlier_split")) if dd <= LEAK_THRESHOLD else keep.append(r))
        kept_by_split[later] = keep

# ---- FIX: stop 'Falling' from dominating the data (train and val; test is left untouched) ----
import random
random.seed(0)
FALL_ID = FINAL_CLASSES.index("Falling")
def is_fall_only(r): return all(b[0] == FALL_ID for b in r["new"])
def source_stem(name): return name.split(".rf.")[0]      # Roboflow appends '.rf.<hash>' to augmented copies of one photo
def n_fall(rs): return sum(1 for r in rs for b in r["new"] if b[0] == FALL_ID)

if DEDUP_TRAIN_EXACT and "train" in kept_by_split:
    seen, keep = set(), []
    for r in kept_by_split["train"]:
        if r["md5"] in seen:
            rejected.append((r, "exact_duplicate_in_train")); continue
        seen.add(r["md5"]); keep.append(r)
    kept_by_split["train"] = keep

if BALANCE_FALLING:
    print("\n=== BALANCING 'Falling' ===")
    for s in ("train", "val"):
        if s not in kept_by_split:
            continue
        rs = kept_by_split[s]
        before = n_fall(rs)
        if s == "train":      # keep ONE augmented copy per source photo for falling-only images
            seen, keep = set(), []
            for r in rs:
                if is_fall_only(r):
                    k = source_stem(r["img"].name)
                    if k in seen:
                        rejected.append((r, "falling_augmented_copy")); continue
                    seen.add(k)
                keep.append(r)
            rs = keep
        other = Counter(b[0] for r in rs for b in r["new"] if b[0] != FALL_ID)
        cap = int(FALLING_CAP_FACTOR * max(other.values()))
        fo = [r for r in rs if is_fall_only(r)]
        random.shuffle(fo)
        cur, drop = n_fall(rs), set()
        for r in fo:
            if cur <= cap:
                break
            drop.add(id(r)); cur -= len(r["new"])
        kept_by_split[s] = [r for r in rs if id(r) not in drop]
        rejected += [(r, "falling_capped") for r in rs if id(r) in drop]
        print(f"  {s:5s}: Falling annotations {before} -> {n_fall(kept_by_split[s])} (cap {cap}, largest other class {max(other.values())})")

# ---- write the new dataset ----
if DST.exists():
    shutil.rmtree(DST)
for s, rs in kept_by_split.items():
    (DST / "images" / s).mkdir(parents=True); (DST / "labels" / s).mkdir(parents=True)
    for r in rs:
        shutil.copy2(r["img"], DST / "images" / s / r["img"].name)
        (DST / "labels" / s / (r["img"].stem + ".txt")).write_text(
            "\n".join(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for c, x, y, w, h in r["new"]) + "\n")

rej_count = Counter((r["split"], why) for r, why in rejected)
print("\n=== CLEANING SUMMARY ===")
for s in SPLITS:
    print(f"  {s:5s}: kept {len(kept_by_split[s])} images")
print("  removed classes      :", removed_classes)
print("  annotations removed  :", dict(stats))
print("  images rejected      :")
for (s, why), n in sorted(rej_count.items()):
    print(f"      {s:5s} {why:34s} {n}")
(Path("/content") / "cleaning_log.json").write_text(json.dumps(
    [{"split": r["split"], "file": r["img"].name, "reason": why} for r, why in rejected], indent=1))
print("  full list saved to /content/cleaning_log.json")
if len(kept_by_split.get("val", [])) < 100 or len(kept_by_split.get("test", [])) < 50:
    print("\nWARNING: val/test are small after cleaning. Lower LEAK_THRESHOLD (e.g. 1) to remove fewer images, "
          "or set REMOVE_LEAKAGE=False (then val/test scores will be optimistic).")


### Cell 6: final `data.yaml` and class balance
**Does:** writes `data.yaml` with exactly 5 classes and prints the final distribution with a verdict (reasonably / moderately / severely imbalanced) and advice. No data is duplicated artificially.
**Expect:** YAML content, a distribution table and a verdict.
**Stop if:** any split has zero samples of a class.

In [ ]:
import yaml
from collections import Counter

DATA_YAML = DST / "data.yaml"
cfg = {"path": str(DST), "train": "images/train", "val": "images/val",
       "names": {i: n for i, n in enumerate(FINAL_CLASSES)}}
if (DST / "images" / "test").exists():
    cfg["test"] = "images/test"
DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(DATA_YAML.read_text())

# ---- final class distribution ----
dist = {}
for s in ("train", "val", "test"):
    d = DST / "labels" / s
    if not d.exists():
        continue
    a, im = Counter(), {c: 0 for c in range(5)}
    for f in d.glob("*.txt"):
        ids = [int(l.split()[0]) for l in f.read_text().splitlines() if l.strip()]
        a.update(ids)
        for c in set(ids):
            im[c] += 1
    dist[s] = (a, im)

print("=== FINAL CLASS DISTRIBUTION (annotations / images) ===")
print(f"  {'class':10s}" + "".join(f"{s:>16s}" for s in dist))
for i, n in enumerate(FINAL_CLASSES):
    print(f"  {i} {n:8s}" + "".join(f"{dist[s][0][i]:>8d}/{dist[s][1][i]:<7d}" for s in dist))

tr = dist["train"][0]
counts = [tr[i] for i in range(5)]
if min(counts) == 0:
    verdict = "SEVERELY IMBALANCED (a class has zero training samples)"
else:
    ratio = max(counts) / min(counts)
    verdict = ("REASONABLY BALANCED" if ratio <= 1.5 else
               "MODERATELY IMBALANCED" if ratio <= 4 else "SEVERELY IMBALANCED")
    print(f"\n  largest/smallest class ratio (train): {ratio:.1f}")
print("  VERDICT:", verdict)
rare = [FINAL_CLASSES[i] for i in range(5) if counts[i] < 0.5 * (sum(counts) / 5)]
if "REASONABLY" not in verdict:
    print(f"  Under-represented: {rare}")
    print("  What to do (no fake duplication was added):")
    print("   - keep the default augmentation (mosaic/scale/colour) in training")
    print("   - judge the model by PER-CLASS recall, not by overall mAP alone")
    print("   - if a rare class stays weak, add real images of it (your own webcam clips) and retrain")
for s in dist:
    miss = [FINAL_CLASSES[i] for i in range(5) if dist[s][0][i] == 0]
    if miss:
        print(f"  WARNING: split '{s}' has NO samples of {miss}")


### Cell 7: visual verification BEFORE training
**Does:** automatic checks (one-to-one images/labels, valid IDs, boxes inside image, all 5 classes in each split) plus 3 examples per class with boxes and class names.
**Expect:** `AUTOMATIC CHECKS: ALL PASSED` and a 5x3 image grid. Go through the checklist printed below the grid.
**Stop if:** boxes or names look wrong. Do not waste GPU time.

In [ ]:
import random, cv2
import matplotlib.pyplot as plt
from pathlib import Path

PALETTE = [(0, 200, 0), (255, 160, 0), (0, 160, 255), (180, 80, 255), (255, 0, 0)]

def load_boxes(lbl):
    return [(int(l.split()[0]), *map(float, l.split()[1:])) for l in Path(lbl).read_text().splitlines() if l.strip()]

def draw_clean(img_path):
    im = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    H, W = im.shape[:2]
    lbl = DST / "labels" / img_path.parent.name / (img_path.stem + ".txt")
    for c, xc, yc, w, h in load_boxes(lbl):
        x1, y1, x2, y2 = int((xc - w / 2) * W), int((yc - h / 2) * H), int((xc + w / 2) * W), int((yc + h / 2) * H)
        cv2.rectangle(im, (x1, y1), (x2, y2), PALETTE[c], 3)
        cv2.putText(im, f"{c}:{FINAL_CLASSES[c]}", (x1, max(y1 - 6, 14)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, PALETTE[c], 2)
    return im

# --- automatic checks over the WHOLE clean dataset ---
problems = []
present = {}
for s in ("train", "val", "test"):
    idir, ldir = DST / "images" / s, DST / "labels" / s
    if not idir.exists():
        continue
    present[s] = set()
    img_stems = {p.stem for p in idir.iterdir()}
    lbl_stems = {p.stem for p in ldir.glob("*.txt")}
    if img_stems != lbl_stems:
        problems.append(f"{s}: images and labels do not match one-to-one")
    for f in ldir.glob("*.txt"):
        for c, xc, yc, w, h in load_boxes(f):
            present[s].add(c)
            if c not in range(5): problems.append(f"{f.name}: bad class {c}")
            if xc - w / 2 < -1e-4 or yc - h / 2 < -1e-4 or xc + w / 2 > 1 + 1e-4 or yc + h / 2 > 1 + 1e-4:
                problems.append(f"{f.name}: box outside image")
for s, ids in present.items():
    if len(ids) < 5:
        problems.append(f"{s}: classes missing {[FINAL_CLASSES[i] for i in range(5) if i not in ids]}")
print("AUTOMATIC CHECKS:", "ALL PASSED" if not problems else "")
for p in problems[:15]:
    print("  FAIL:", p)

# --- 3 random examples per class (train) ---
by_class = {c: [] for c in range(5)}
for f in (DST / "labels" / "train").glob("*.txt"):
    for c in {b[0] for b in load_boxes(f)}:
        by_class[c].append(f)
fig, axs = plt.subplots(5, 3, figsize=(13, 20))
for c in range(5):
    pick = random.sample(by_class[c], min(3, len(by_class[c])))
    for j in range(3):
        axs[c][j].axis("off")
        if j < len(pick):
            img = next((DST / "images" / "train" / (pick[j].stem + e) for e in (".jpg", ".jpeg", ".png", ".bmp", ".webp")
                        if (DST / "images" / "train" / (pick[j].stem + e)).exists()))
            axs[c][j].imshow(draw_clean(img)); axs[c][j].set_title(f"class {c} = {FINAL_CLASSES[c]}", fontsize=10)
plt.tight_layout(); plt.show()

print("""
MANUAL CHECKLIST (look at the images above):
  [ ] Is every box around the correct person?
  [ ] Does the class name match what the person is doing?
      (Sleeping = lying down, Falling = mid-fall / on the floor after a fall)
  [ ] Are there any unwanted classes left? (should be none: only 5 names exist)
  [ ] Are all boxes inside the image?
  [ ] Are all five classes shown?
If many boxes/labels look wrong, STOP and fix the mapping in Cell 5 before training.""")


### Cell 8: load pretrained YOLO11
**Does:** loads `yolo11s.pt` (COCO pretrained). Small is more accurate than nano on similar poses (this fixes many false Falling). Switch to `yolo11n.pt` if you need the fastest laptop CPU speed.
**Expect:** `Loaded: yolo11s.pt | GPU: Tesla T4`.

In [ ]:
from ultralytics import YOLO
import torch

MODEL_NAME = "yolo11s.pt"      # FIX: small model separates similar poses (sitting vs falling) better than nano. Use "yolo11n.pt" only if you need max speed on CPU
model = YOLO(MODEL_NAME)       # downloads COCO-pretrained weights
print("Loaded:", MODEL_NAME, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")


### Cell 9: train
**Why these settings:** `epochs=50` (fine-tuning a pretrained model converges in 30-50; early stopping with `patience=15` ends sooner), `imgsz=640` (standard, keeps small/far people visible), `batch=32` (use 16 on out-of-memory), `cls=1.0` (stronger classification loss), `optimizer=auto` (Ultralytics picks optimizer and learning rate), `flipud=0` (never flip vertically: it would create fake falls), mosaic off for the last 10 epochs.
**Expect:** an epoch table (~10-25 min total). mAP50 should rise and flatten.
**Result:** best weights at `/content/runs/vital_yolo11s/weights/best.pt` (best validation epoch, not the last).

In [ ]:
results = model.train(
    data=str(DATA_YAML),
    epochs=50,            # 40-60 is enough for fine-tuning; early stopping ends it sooner
    patience=15,          # stop if val metrics do not improve for 15 epochs
    imgsz=640,
    batch=32,             # nano model fits easily on a T4; lower to 16 if you get an out-of-memory error
    optimizer="auto",     # Ultralytics picks AdamW/SGD and the learning rate itself
    cls=1.0,              # FIX: classification loss gain (default 0.5). Pushes the model to separate Sitting/Standing/Falling
    erasing=0.4,          # random erasing: model must not rely on one body part, fewer wrong Falling calls
    cos_lr=True,          # smooth LR decay
    close_mosaic=10,      # last 10 epochs without mosaic -> cleaner final weights
    # --- augmentation (posture-safe) ---
    fliplr=0.5,           # left-right flip is fine
    flipud=0.0,           # NEVER up-down flip: an upside-down person looks like a fall
    degrees=5, translate=0.1, scale=0.5,
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.4,
    mosaic=1.0, mixup=0.0,
    # --- housekeeping ---
    device=0, workers=2, seed=0,
    project="/content/runs", name="vital_yolo11s", exist_ok=True,
    val=True, plots=True,
)
BEST_PT = Path(model.trainer.save_dir) / "weights" / "best.pt"   # best VALIDATION checkpoint, not last.pt
print("\nBest model:", BEST_PT, "| exists:", BEST_PT.exists())


### Cell 10: validate best.pt (val and test)
**Does:** prints Precision, Recall, mAP50, mAP50-95 overall and per class on val and on the untouched test split.
**Expect:** two tables. Check per-class recall, especially **Falling**.

In [ ]:
from ultralytics import YOLO
best = YOLO(str(BEST_PT))
METRICS = {}
for split in ("val", "test"):
    if split not in cfg and split != "val":
        continue
    m = best.val(data=str(DATA_YAML), split=split, imgsz=640, batch=32, device=0, plots=(split == "val"), verbose=False)
    METRICS[split] = m
    print(f"\n===== {split.upper()} SET =====")
    print(f"Precision {m.box.mp:.3f} | Recall {m.box.mr:.3f} | mAP50 {m.box.map50:.3f} | mAP50-95 {m.box.map:.3f}")
    print(f"  {'class':10s}{'P':>8s}{'R':>8s}{'mAP50':>8s}{'mAP50-95':>10s}")
    for k, ci in enumerate(m.box.ap_class_index):
        p, r, a50, a = m.box.class_result(k)
        print(f"  {FINAL_CLASSES[ci]:10s}{p:8.3f}{r:8.3f}{a50:8.3f}{a:10.3f}")
print("\nTest set = images the model never saw during training or checkpoint selection. Trust the TEST numbers more than val.")


### Cell 11: confusion matrix and risky confusions
**Does:** shows the confusion matrix and prints the rates for Falling vs Sitting, Sitting vs Standing, Sleeping vs Sitting, Walking vs Standing, plus missed detections and the weakest classes with likely reasons.
**Expect:** images and percentages. Completed training does not mean the model is good: judge by these numbers.

In [ ]:
import numpy as np
from IPython.display import Image as IPImage, display

val_dir = Path(METRICS["val"].save_dir)
for f in ("confusion_matrix_normalized.png", "confusion_matrix.png"):
    if (val_dir / f).exists():
        display(IPImage(str(val_dir / f), width=650))

# rows = predicted, columns = true, last index = background (ultralytics convention)
cm = np.array(METRICS["test" if "test" in METRICS else "val"].confusion_matrix.matrix)
def rate(true_n, pred_n):
    t, p = FINAL_CLASSES.index(true_n), FINAL_CLASSES.index(pred_n)
    tot = cm[:, t].sum()
    return 100 * cm[p, t] / tot if tot else float("nan")

print("\nSAFETY-CRITICAL CONFUSIONS (% of TRUE class predicted as the other):")
for a, b in [("Falling", "Sitting"), ("Sitting", "Standing"), ("Sleeping", "Sitting"), ("Walking", "Standing")]:
    print(f"  true {a:9s} -> predicted {b:9s}: {rate(a, b):5.1f}%    |  true {b:9s} -> predicted {a:9s}: {rate(b, a):5.1f}%")
print("\nMISSED (true object predicted as background):")
for i, n in enumerate(FINAL_CLASSES):
    tot = cm[:, i].sum()
    print(f"  {n:9s}: {100 * cm[5, i] / tot if tot else float('nan'):5.1f}%")

per = {FINAL_CLASSES[ci]: METRICS['val'].box.class_result(k)[3] for k, ci in enumerate(METRICS['val'].box.ap_class_index)}
weak = sorted(per, key=per.get)[:2]
print("\nWEAKEST classes by mAP50-95:", [(w, round(per[w], 3)) for w in weak])
WHY = {"Falling": "rare and brief pose; looks like Sleeping/Sitting when the person is already on the floor",
       "Sleeping": "lying pose is similar to Falling; bed/sofa occlusion hides the body",
       "Sitting": "legs hidden by furniture; similar silhouette to Standing at low resolution",
       "Standing": "similar silhouette to Walking in a single frame (walking needs motion)",
       "Walking": "one still frame cannot show motion; confused with Standing"}
for w in weak:
    print(f"  {w}: {WHY[w]}")
print("\nDo NOT call the model 'accurate' unless Falling recall is high on the TEST set and Falling->Sitting/Sleeping errors are low.")


# ---- FIX: FALSE FALL ALARM RATE = how often a NON-falling person is predicted as Falling ----
F = FINAL_CLASSES.index("Falling")
print("\nFALSE FALL ALARMS (true class -> predicted Falling):")
worst = 0
for i, n in enumerate(FINAL_CLASSES):
    if i == F:
        continue
    tot = cm[:, i].sum()
    v = 100 * cm[F, i] / tot if tot else float("nan")
    worst = max(worst, 0 if v != v else v)
    print(f"  true {n:9s} predicted as Falling: {v:5.1f}%")
print("  -> GOOD if every value above is under ~5%. If not: lower FALLING_CAP_FACTOR in Cell 5 (e.g. 0.7) and retrain,")
print("     and use the Falling threshold from the next cell.")


### Cell 11b: pick the Falling confidence threshold (NEW)
**Does:** finds the confidence above which a **Falling** prediction is right at least 90% of the time on the validation set, and saves it to `vital_config.json`. Use this value (not a random 0.35) in your app.
**Expect:** a threshold between roughly 0.5 and 0.8 and the recall you keep at that threshold.

In [ ]:
import json, numpy as np
mv = METRICS["val"].box
k = list(mv.ap_class_index).index(FINAL_CLASSES.index("Falling"))
px, p_curve, r_curve = np.array(mv.px), np.array(mv.p_curve[k]), np.array(mv.r_curve[k])
TARGET_PRECISION = 0.90          # raise to 0.95 for even fewer false alarms (you will miss more real falls)
ok = np.where(p_curve >= TARGET_PRECISION)[0]
FALL_CONF = float(px[ok[0]]) if len(ok) else 0.70
FALL_CONF = float(np.clip(FALL_CONF, 0.50, 0.85))
j = int(np.argmin(np.abs(px - FALL_CONF)))
print(f"Falling threshold (precision >= {TARGET_PRECISION:.0%}): {FALL_CONF:.2f}")
print(f"  at this threshold: precision {p_curve[j]:.2f}, recall {r_curve[j]:.2f}")
json.dump({"classes": FINAL_CLASSES, "fall_conf": round(FALL_CONF, 2), "general_conf": 0.30,
           "imgsz": 640}, open("/content/vital_config.json", "w"), indent=1)
print("saved /content/vital_config.json")

### Cell 12: test predictions
**Does:** predicts on 8 random test images.
**Expect:** a grid with boxes, class names and confidences.

In [ ]:
import random, glob
import matplotlib.pyplot as plt, cv2

src = "test" if (DST / "images" / "test").exists() else "val"
files = random.sample(sorted(glob.glob(str(DST / "images" / src / "*"))), 8)
preds = best.predict(files, conf=0.35, imgsz=640, device=0, agnostic_nms=True, verbose=False)   # agnostic_nms: one class per person
fig, axs = plt.subplots(2, 4, figsize=(18, 9))
for ax, r in zip(axs.flat, preds):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis("off")
plt.tight_layout(); plt.show()


### Cell 12b: temporal filter for video / webcam (NEW, the main fix for 'falling again and again')
**Why:** one frame cannot tell *sitting down / standing up* from *falling*. In video the model flickers to Falling for a few frames. This cell writes `vital_infer.py`, which tracks each person and only confirms a fall when the **sequence** matches: upright -> quickly lying (wide box) -> Falling predicted consistently. It raises **one** alert per fall, not one per frame.
**Expect:** the file is written. Use it in the next cell and on your laptop.

In [ ]:
%%writefile /content/vital_infer.py
"""
VITAL inference helper: turns noisy per-frame YOLO predictions into stable activities.

WHY: a detector only sees ONE frame. During sit->stand, a bent torso looks like a fall for a
few frames, so raw output flickers to "Falling". A real fall is a SEQUENCE:
    upright person  ->  quickly lying (wide box)  ->  stays down
This filter checks that sequence per tracked person and raises ONE fall alert (not one per frame).

Usage (laptop or Colab):
    python vital_infer.py --weights best.pt --source 0                 # webcam
    python vital_infer.py --weights best.pt --source video.mp4 --save out.mp4

Needs: pip install ultralytics   (the filter class itself is pure Python)
"""
from collections import deque, defaultdict
from dataclasses import dataclass
import statistics

CLASSES = ["Walking", "Standing", "Sitting", "Sleeping", "Falling"]


@dataclass
class FilterConfig:
    fps: float = 25.0           # camera / video fps (used to convert seconds to frames)
    window_s: float = 1.5       # smoothing window for the activity label
    history_s: float = 4.0      # how far back we look for "was this person upright?"
    fall_conf: float = 0.60     # a frame only counts as "falling" if its confidence >= this
    fall_ratio: float = 0.60    # >= 60% of frames in the window must say Falling
    min_frames: int = 6         # need at least this many frames in the window to decide
    min_aspect: float = 1.00    # width/height of box to count as "lying" (1.0 = wider than tall)
    upright_aspect: float = 0.80  # width/height <= this means upright (standing/sitting/walking)
    hold_s: float = 3.0         # keep showing "Falling" this long after the alert
    cooldown_s: float = 8.0     # never alert twice for the same person within this time
    forget_s: float = 3.0       # drop a track that has not been seen for this long


class _Track:
    def __init__(self):
        self.hist = deque()            # (t, cls, conf, aspect)
        self.alert_until = -1e9
        self.last_alert = -1e9
        self.armed = True              # re-armed when the person is seen upright again
        self.last_nonfall = "Standing"
        self.last_seen = 0.0


class ActivityFilter:
    def __init__(self, cfg: FilterConfig = None):
        self.cfg = cfg or FilterConfig()
        self.tracks = defaultdict(_Track)

    def update(self, track_id, t, cls_name, conf, box_xyxy):
        """One detection of one person at time t (seconds).
        Returns (label, new_fall_alert: bool). new_fall_alert is True on ONE frame only."""
        c = self.cfg
        x1, y1, x2, y2 = box_xyxy
        aspect = max(x2 - x1, 1e-6) / max(y2 - y1, 1e-6)
        st = self.tracks[track_id]
        st.last_seen = t
        st.hist.append((t, cls_name, float(conf), aspect))
        while st.hist and t - st.hist[0][0] > c.history_s:
            st.hist.popleft()
        win = [h for h in st.hist if t - h[0] <= c.window_s]
        n = len(win)

        med_aspect = statistics.median(h[3] for h in win)
        fall_frames = sum(1 for h in win if h[1] == "Falling" and h[2] >= c.fall_conf)
        upright_seen = any(h[3] <= c.upright_aspect for h in st.hist)

        # person is clearly upright again -> allow a future alert
        if n >= c.min_frames and med_aspect <= c.upright_aspect and t > st.alert_until:
            st.armed = True

        new_alert = False
        if (n >= c.min_frames
                and fall_frames / n >= c.fall_ratio       # Falling predicted consistently, not a flicker
                and med_aspect >= c.min_aspect            # body is now horizontal
                and upright_seen                          # ...and was upright a moment ago
                and st.armed
                and t - st.last_alert >= c.cooldown_s):
            new_alert = True
            st.last_alert = t
            st.alert_until = t + c.hold_s
            st.armed = False

        if t <= st.alert_until:
            return "Falling", new_alert

        # Not a confirmed fall: best NON-falling class by summed confidence in the window
        score = defaultdict(float)
        for _, cl, cf, _a in win:
            if cl != "Falling":
                score[cl] += cf
        if score:
            label = max(score, key=score.get)
            st.last_nonfall = label
        elif med_aspect >= c.min_aspect:
            label = "Sleeping"                            # lying, but no fall sequence seen
        else:
            label = st.last_nonfall
        return label, False

    def cleanup(self, t):
        for k in [k for k, s in self.tracks.items() if t - s.last_seen > self.cfg.forget_s]:
            del self.tracks[k]


def run(weights, source, conf=0.30, imgsz=640, fall_conf=0.60, save=None, show=True, min_aspect=1.0):
    import cv2
    from ultralytics import YOLO
    model = YOLO(weights)
    cap = cv2.VideoCapture(0 if str(source) == "0" else str(source))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    flt = ActivityFilter(FilterConfig(fps=fps, fall_conf=fall_conf, min_aspect=min_aspect))
    writer = None
    frame_i = 0
    COL = {"Walking": (0, 200, 0), "Standing": (255, 160, 0), "Sitting": (0, 160, 255),
           "Sleeping": (180, 80, 255), "Falling": (0, 0, 255)}
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        t = frame_i / fps
        frame_i += 1
        # agnostic_nms=True -> ONE class per person; track() gives a stable ID per person
        r = model.track(frame, persist=True, conf=conf, imgsz=imgsz, agnostic_nms=True,
                        tracker="bytetrack.yaml", verbose=False)[0]
        if r.boxes is not None and r.boxes.id is not None:
            for box, tid, cl, cf in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.int().cpu().tolist(),
                                        r.boxes.cls.int().cpu().tolist(), r.boxes.conf.cpu().tolist()):
                label, alert = flt.update(tid, t, CLASSES[cl], cf, box)
                x1, y1, x2, y2 = map(int, box)
                cv2.rectangle(frame, (x1, y1), (x2, y2), COL[label], 3 if label == "Falling" else 2)
                cv2.putText(frame, f"#{tid} {label}", (x1, max(y1 - 8, 16)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, COL[label], 2)
                if alert:
                    print(f"[ALERT] fall confirmed for person #{tid} at {t:.1f}s")
        flt.cleanup(t)
        if save:
            if writer is None:
                h, w = frame.shape[:2]
                writer = cv2.VideoWriter(save, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
            writer.write(frame)
        if show:
            cv2.imshow("VITAL", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    cap.release()
    if writer:
        writer.release()
    if show:
        cv2.destroyAllWindows()


if __name__ == "__main__":
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--source", default="0")
    ap.add_argument("--conf", type=float, default=0.30)
    ap.add_argument("--fall-conf", type=float, default=0.60)
    ap.add_argument("--min-aspect", type=float, default=1.0,
                    help="width/height of box that counts as lying. Lower (0.8) for overhead/angled cameras")
    ap.add_argument("--save", default=None)
    ap.add_argument("--no-show", action="store_true")
    a = ap.parse_args()
    run(a.weights, a.source, a.conf, 640, a.fall_conf, a.save, not a.no_show, a.min_aspect)


### Cell 12c: test on your own video (NEW, optional but recommended)
**Does:** runs the model with the temporal filter on a video, e.g. you sitting and standing several times. Set `VIDEO` to a file you upload to Colab (use a clip of sitting/standing/walking). 
**Expect:** `[ALERT]` lines only for real falls; the output video `/content/vital_out.mp4` should show Sitting/Standing, not Falling, during sit<->stand.

In [ ]:
import sys, json
sys.path.insert(0, "/content")
import importlib, vital_infer
importlib.reload(vital_infer)

VIDEO = "/content/test_video.mp4"      # EDIT: upload a video to Colab and put its path here
cfg_json = json.load(open("/content/vital_config.json"))
if not Path(VIDEO).exists():
    print("Upload a test video to", VIDEO, "then re-run this cell.")
else:
    vital_infer.run(str(BEST_PT), VIDEO, conf=cfg_json["general_conf"], fall_conf=cfg_json["fall_conf"],
                    save="/content/vital_out.mp4", show=False)
    print("Saved /content/vital_out.mp4 (download it from the Files panel to watch)")

### Cell 13: download best.pt
**Does:** saves a Drive backup (if mounted) and downloads `best.pt`, `vital_config.json` and `vital_infer.py` to your laptop.
**Expect:** three browser downloads. Put `best.pt` in `VITAL/models/`. In your app, import `ActivityFilter` from `vital_infer.py` and feed it every detection (see its docstring), or run `python vital_infer.py --weights best.pt --source 0`.

In [ ]:
import shutil
from google.colab import files

print("best.pt is at:", BEST_PT, f"({BEST_PT.stat().st_size / 1e6:.1f} MB)")
if Path("/content/drive/MyDrive").exists():           # backup copy in Drive if it is mounted
    shutil.copy(BEST_PT, "/content/drive/MyDrive/VITAL_best.pt")
    print("Backup copy: Google Drive > My Drive > VITAL_best.pt")
files.download(str(BEST_PT))                          # browser download to your laptop
files.download("/content/vital_config.json")          # tuned Falling threshold
files.download("/content/vital_infer.py")             # temporal filter + video/webcam runner (use this in your app)
